# Eksperyment C - mixed training synthetic + real

Notebook do odpalenia eksperymentu C: trening na synthetic + mala porcja real train/val, a potem ewaluacja na realnym holdoucie.

Tryby:
- `USE_SMOKE = True`: szybki test techniczny, `synthetic_1k`, tylko `1% real`, 3 epoki.
- `USE_SMOKE = False`: finalny eksperyment, `synthetic_10k`, warianty `1/5/10/25% real`, 60 epok.

Wyniki ida do `results/per_size/expC*_ml.json`, `results/baselines/expC*_ml.json` oraz `results/expC_mixed_summary.{csv,md}`.

In [ ]:
# 0. GPU sanity check
# Jesli ta komorka nie przejdzie: Runtime -> Change runtime type -> GPU.
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

import torch
assert torch.cuda.is_available(), "Brak GPU. Wlacz Runtime -> Change runtime type -> GPU."
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 1. Repo + zaleznosci
# Zmien REPO_URL/REPO_BRANCH, jesli pracujesz na innym forku albo branchu.
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/milosz-amg/rareplanes-domain-shift.git"
REPO_BRANCH = ""
REPO_NAME = "rareplanes-domain-shift"

# Stabilny katalog bazowy: komorka moze byc odpalana wiele razy bez klonowania repo w repo.
BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
if Path.cwd().name == REPO_NAME:
    BASE_DIR = Path.cwd().parent
REPO_DIR = BASE_DIR / REPO_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(BASE_DIR)

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
print("cwd =", Path.cwd())

if REPO_BRANCH:
    subprocess.check_call(["git", "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "checkout", REPO_BRANCH])

# Nie przerywamy notebooka, jesli lokalnie sa jakies zmiany albo pull nie jest fast-forward.
subprocess.run(["git", "pull", "--ff-only"], check=False)

required = [
    "src/make_mixed_dataset.py",
    "src/sweep_C_mixed.sh",
    "src/train_yolo.py",
    "src/eval_per_size.py",
]
missing = [p for p in required if not Path(p).exists()]
assert not missing, "Brakuje skryptow eksperymentu C: " + ", ".join(missing)

packages = [
    "ultralytics",
    "pycocotools",
    "pillow",
    "numpy",
    "pandas",
    "matplotlib",
    "tabulate",
    "opencv-python-headless",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-cache-dir", "-q", *packages])
print("OK: repo i zaleznosci gotowe")

## Konfiguracja smoke/final

Najpierw odpal `USE_SMOKE = True`. Gdy przejdzie bez bledow, zmien na `False` i odpal finalny eksperyment.

In [ ]:
# 2. Konfiguracja eksperymentu C
USE_SMOKE = True  # True: szybki test; False: finalny run

if USE_SMOKE:
    SRC_DATASET = "data/yolo/synthetic_1k"
    DATASET_TAG = "1k_smoke"
    EPOCHS = 3
    BATCH = 32
    WORKERS = 2
    DEVICE = 0
    PCTS = "1"
    FRACS = "0.01"
else:
    SRC_DATASET = "data/yolo/synthetic_10k"
    DATASET_TAG = "10k"
    EPOCHS = 30
    BATCH = 32
    WORKERS = 2
    DEVICE = 0
    PCTS = "1 5 10 25"
    FRACS = "0.01 0.05 0.10 0.25"

IMG_SIZE = 512
MODEL = "yolov10n.pt"

print({
    "USE_SMOKE": USE_SMOKE,
    "SRC_DATASET": SRC_DATASET,
    "DATASET_TAG": DATASET_TAG,
    "EPOCHS": EPOCHS,
    "BATCH": BATCH,
    "WORKERS": WORKERS,
    "DEVICE": DEVICE,
    "PCTS": PCTS,
    "FRACS": FRACS,
    "IMG_SIZE": IMG_SIZE,
    "MODEL": MODEL,
})

## Dane dla eksperymentu C

C potrzebuje real train/val do mixed training oraz real test do ewaluacji. Pobieranie jest wznawialne: istniejące pliki nie są pobierane ponownie.

In [ ]:
# 3. Adnotacje + real train/test tiles
import os
import tarfile
import urllib.request
from pathlib import Path

B = "https://rareplanes-public.s3.amazonaws.com"
for d in [
    "data/real/annotations",
    "data/real/tarballs",
    "data/synthetic/annotations",
    "data/synthetic/images/train",
    "data/real/PS-RGB_tiled",
]:
    Path(d).mkdir(parents=True, exist_ok=True)

def download(url, dst):
    dst = Path(dst)
    if dst.exists() and dst.stat().st_size > 0:
        print("[skip]", dst)
        return
    part = dst.with_name(dst.name + ".part")
    if part.exists():
        part.unlink()
    print("[download]", url, "->", dst)
    urllib.request.urlretrieve(url, part)
    os.replace(part, dst)

def safe_extract(tar_path, dst_dir):
    tar_path = Path(tar_path)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_resolved = dst_dir.resolve()
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar.getmembers():
            target = (dst_dir / member.name).resolve()
            target.relative_to(dst_resolved)
        tar.extractall(dst_dir)

for f in ["instances_train_aircraft.json", "instances_test_aircraft.json"]:
    download(f"{B}/real/metadata_annotations/{f}", f"data/real/annotations/{f}")
    download(f"{B}/synthetic/metadata_annotations/{f}", f"data/synthetic/annotations/{f}")

download(f"{B}/real/tarballs/train/RarePlanes_train_PS-RGB_tiled.tar.gz", "data/real/tarballs/train.tar.gz")
download(f"{B}/real/tarballs/test/RarePlanes_test_PS-RGB_tiled.tar.gz", "data/real/tarballs/test.tar.gz")

real_tile_dir = Path("data/real/PS-RGB_tiled/PS-RGB_tiled")
real_tiles_before = len(list(real_tile_dir.glob("*.png"))) if real_tile_dir.exists() else 0
if real_tiles_before > 7000:
    print("[skip extract] real tiles:", real_tiles_before)
else:
    safe_extract("data/real/tarballs/train.tar.gz", "data/real/PS-RGB_tiled")
    safe_extract("data/real/tarballs/test.tar.gz", "data/real/PS-RGB_tiled")

real_tiles = len(list(real_tile_dir.glob("*.png")))
print("real tiles:", real_tiles)
assert real_tiles > 7000, "Za malo real tiles - sprawdz pobieranie tarballi."

In [ ]:
# 4. Synthetic 10k przez HTTPS z resume
import os
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DST = Path("data/synthetic/images/train")
DST.mkdir(parents=True, exist_ok=True)
files = [l.strip() for l in open("configs/synthetic_10k_train_files.txt")]
files += [l.strip() for l in open("configs/synthetic_10k_val_files.txt")]
files = [f for f in files if f]
print("do pobrania/sprawdzenia:", len(files))

def fetch(fn):
    dst = DST / fn
    part = DST / (fn + ".part")
    if dst.exists() and dst.stat().st_size > 0:
        return True
    try:
        urllib.request.urlretrieve(f"{B}/synthetic/train/images/{fn}", part)
        os.replace(part, dst)
        return True
    except Exception:
        if part.exists():
            part.unlink()
        return False

ok = 0
with ThreadPoolExecutor(max_workers=32) as ex:
    for i, r in enumerate(ex.map(fetch, files), 1):
        ok += int(r)
        if i % 1000 == 0 or i == len(files):
            print(f"  {i}/{len(files)} ok={ok}")

have_selected = sum(1 for fn in files if (DST / fn).exists() and (DST / fn).stat().st_size > 0)
print(f"GOTOWE: selected_ok={have_selected}/{len(files)}")
assert have_selected >= int(len(files) * 0.99), "Za malo synthetic - uruchom komorke ponownie, resume dociagnie reszte."

In [ ]:
# 5. COCO -> YOLO + subsety synthetic + real_aircraft
!python3 src/coco_to_yolo.py --domain synthetic --classes aircraft --val-frac 0.15
!python3 src/coco_to_yolo.py --domain real --classes aircraft --val-frac 0.15
!python3 src/make_subset.py --n-train 10000 --name synthetic_10k
!python3 src/make_subset.py --n-train 1000 --name synthetic_1k

from pathlib import Path
counts = {
    "synthetic_10k_train": len(list(Path("data/yolo/synthetic_10k/images/train").glob("*.png"))),
    "synthetic_1k_train": len(list(Path("data/yolo/synthetic_1k/images/train").glob("*.png"))),
    "real_train": len(list(Path("data/yolo/real_aircraft/images/train").glob("*.png"))),
    "real_val": len(list(Path("data/yolo/real_aircraft/images/val").glob("*.png"))),
    "real_test": len(list(Path("data/yolo/real_aircraft/images/test").glob("*.png"))),
}
print(counts)
assert counts["synthetic_1k_train"] >= 900, "synthetic_1k nie wyglada OK"
assert counts["synthetic_10k_train"] >= 9900, "synthetic_10k nie wyglada OK"
assert counts["real_train"] > 0 and counts["real_val"] > 0, "real_aircraft nie wyglada OK"

In [ ]:
# 6. Sweep C: smoke albo final, zależnie od USE_SMOKE
print("Odpalam sweep C z konfiguracja:")
print({
    "SRC_DATASET": SRC_DATASET,
    "DATASET_TAG": DATASET_TAG,
    "EPOCHS": EPOCHS,
    "BATCH": BATCH,
    "WORKERS": WORKERS,
    "DEVICE": DEVICE,
    "PCTS": PCTS,
    "FRACS": FRACS,
    "IMG_SIZE": IMG_SIZE,
    "MODEL": MODEL,
})

!SRC_DATASET={SRC_DATASET} DATASET_TAG={DATASET_TAG} EPOCHS={EPOCHS} BATCH={BATCH} WORKERS={WORKERS} DEVICE={DEVICE} PCTS="{PCTS}" FRACS="{FRACS}" IMG_SIZE={IMG_SIZE} MODEL={MODEL} bash src/sweep_C_mixed.sh

In [ ]:
# 7. Tabela wynikow C z JSON-ow
import json
from pathlib import Path
import pandas as pd

rows = []
for p in sorted(Path("results/per_size").glob("expC*_ml.json")):
    d = json.load(open(p))
    m = d.get("metrics", {})
    rows.append({
        "run": d.get("name", p.stem),
        "mAP@50": m.get("AP@.5"),
        "mAP@50:95": m.get("AP@[.5:.95]"),
        "AP_S": m.get("AP_small"),
        "AP_M": m.get("AP_medium"),
        "AP_L": m.get("AP_large"),
        "AR@100": m.get("AR@100"),
        "n_det": d.get("n_detections"),
        "file": str(p),
    })

df = pd.DataFrame(rows)
display(df)
Path("results").mkdir(exist_ok=True)
df.to_csv("results/expC_mixed_summary.csv", index=False)
df.to_markdown("results/expC_mixed_summary.md", index=False)
print("zapisano: results/expC_mixed_summary.csv/md")

In [ ]:
# 8. Pobranie wynikow C na komputer
from pathlib import Path
import zipfile

try:
    from google.colab import files as colab_files
except Exception:
    colab_files = None

targets = []
targets += list(Path("results/per_size").glob("expC*_ml.json"))
targets += list(Path("results/baselines").glob("expC*_ml.json"))
for extra in ["results/expC_mixed_summary.csv", "results/expC_mixed_summary.md"]:
    p = Path(extra)
    if p.exists():
        targets.append(p)

zip_path = Path("expC_mixed_results.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    for p in targets:
        z.write(p, p.as_posix())

print("pliki w paczce:")
for p in targets:
    print(" -", p)

if colab_files is not None:
    colab_files.download(str(zip_path))
else:
    print("zip zapisany lokalnie:", zip_path.resolve())